# Data cleaning and preparation
COM747 DS&ML CW2 — A Machine Learning Framework for Detecting Multidimensional Care-Intensity Discordance and Predicting 30-Day Hospital Readmission in Patients with Diabetes.

**Purpose:** create an eligible discharge cohort, deterministic clinical features and reproducible patient-level train/validation/test partitions for RQ1 (adjusted association) and RQ2 (incremental predictive value). This follows the existing `eda/eda.ipynb`. No fitted imputation, scaling, oversampling or expected-care regression occurs here.

**Run:** select Python (DSML CW2), restart kernel, Run All. Place `diabetic_data.csv` and preferably `IDS_mapping.csv` in `data/raw/` relative to the repository root. Run from root or `cleaning/`. Output files are local research data; do not commit them to a public repository. All counts are generated from the actual input, not assumed.

Sources: [UCI dataset and licence](https://doi.org/10.24432/C5230J), [Strack et al. (2014)](https://doi.org/10.1155/2014/781670), [published preprocessing description for disposition codes](https://ogreyesp.github.io/Early_readmission_diabetic_patients/).

In [4]:
from pathlib import Path
import csv, hashlib, json, platform, re
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display
import sys

SEED = 42
candidates = [Path.cwd(), Path.cwd().parent]
root = next((p for p in candidates if (p / "data/raw/diabetic_data.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Place diabetic_data.csv in the project data/raw/ folder; run from root or cleaning/.")
RAW = root / "data/raw/diabetic_data.csv"
OUT = root / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)
print("Kernel:", sys.executable, "| Project:", root)

def sha256(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else hashlib.sha256(f.read()).hexdigest()

raw = pd.read_csv(RAW, keep_default_na=False, na_values=["?", ""],
                  dtype={"encounter_id":"string", "patient_nbr":"string", "diag_1":"string", "diag_2":"string", "diag_3":"string"})
print("Raw shape:", raw.shape)

Kernel: /Users/monishachandrashekar/Desktop/DS&ML-CW2 /.venv/bin/python | Project: /Users/monishachandrashekar/Desktop/DS&ML-CW2 /Data-science-and-Machine-Learning-Course-work
Raw shape: (101766, 50)


/var/folders/2m/b7dfp3r133v8frt6g9kbg0v80000gn/T/ipykernel_5921/1560114629.py:25: DtypeWarning: Columns (10: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(RAW, keep_default_na=False, na_values=["?", ""],


## 1. Schema and identity checks
An encounter is not a unique patient. Remove only exact duplicate rows; conflicting duplicate encounter IDs are a hard error requiring source review. Check all required fields and valid labels before filtering. No silent coercion of invalid numeric values.

In [5]:
CARE = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications"]
COUNTS = ["number_diagnoses", "number_inpatient", "number_emergency", "number_outpatient"]
DEMOGRAPHICS = ["age", "gender", "race"]
ADMISSION = ["admission_type_id", "admission_source_id"]
DIAG = ["diag_1", "diag_2", "diag_3"]
LABS = ["A1Cresult", "max_glu_serum"]
IDS = ["encounter_id", "patient_nbr"]
REQUIRED = IDS + CARE + COUNTS + DEMOGRAPHICS + ADMISSION + DIAG + LABS + ["discharge_disposition_id", "readmitted", "change", "diabetesMed"]
missing_cols = sorted(set(REQUIRED) - set(raw.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = raw.copy()
# Trim text, including whitespace around missing markers; preserve literal 'None'.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip().replace({"?": pd.NA, "": pd.NA})
flow = [{"stage":"raw", "removed":0, "remaining":len(df)}]
def record(stage, before, after):
    flow.append({"stage":stage, "removed":before-after, "remaining":after})

before = len(df)
df = df.drop_duplicates().copy()
record("exact duplicate rows", before, len(df))
if df[IDS].isna().any().any():
    raise ValueError("Missing encounter or patient identifiers; investigate source before splitting.")
if df["encounter_id"].duplicated().any():
    raise ValueError("Conflicting encounter IDs remain after exact deduplication.")
if not df["readmitted"].isin(["NO", ">30", "<30"]).all():
    raise ValueError("Unexpected or missing target labels; investigate raw data.")

for col in CARE + COUNTS + ADMISSION + ["discharge_disposition_id"]:
    converted = pd.to_numeric(df[col], errors="coerce")
    if (df[col].notna() & converted.isna()).any() or (converted.dropna() % 1 != 0).any() or (converted.dropna() < 0).any():
        raise ValueError(f"Invalid non-negative integer values in {col}")
    df[col] = converted.astype("Int64")
print("Schema and ID checks passed.")

Schema and ID checks passed.


## 2. Define eligible discharge cohort
Exclude death or hospice dispositions, as in Strack et al. (2014). The published dataset preprocessing lists IDs 11, 13, 14, 19, 20 and 21; cross-check with the supplied mapping when available. Missing disposition IDs are excluded because eligibility cannot be established. Recorded unknown/invalid disposition categories are retained if not explicit death/hospice; document sensitivity analysis excluding these categories later. Transfers remain included; this is not necessarily discharge home. All age groups retained. Require the four observed care dimensions to be present and plausible; report exclusions, not arbitrary outlier trimming. No exclusion based on demographic missingness or target value.

In [6]:
excluded_codes = {11, 13, 14, 19, 20, 21}
map_path = next((p for p in [RAW.parent/"IDS_mapping.csv", RAW.parent/"IDs_mapping.csv"] if p.is_file()), None)
disposition = {}
if map_path:
    section = None
    with map_path.open(encoding="utf-8-sig", newline="") as f:
        for row in csv.reader(f):
            if not row or not row[0].strip():
                continue
            key = row[0].strip()
            if key.endswith("_id"):
                section = key
            elif section == "discharge_disposition_id" and key.isdigit() and len(row) > 1:
                disposition[int(key)] = row[1].strip()
    if not disposition:
        raise ValueError("Mapping file found, but discharge disposition section could not be parsed.")
    mapped_exclusions = {k for k,v in disposition.items() if re.search(r"expired|hospice|death|died", v, re.I)}
    if mapped_exclusions != excluded_codes:
        raise ValueError(f"Mapping death/hospice IDs {sorted(mapped_exclusions)} differ from expected {sorted(excluded_codes)}; review before proceeding.")
    unmapped = set(df["discharge_disposition_id"].dropna().astype(int)) - set(disposition)
    if unmapped:
        raise ValueError(f"Unmapped discharge disposition IDs: {sorted(unmapped)}")
    print("Death/hospice mapping verified:", {k:disposition[k] for k in sorted(excluded_codes)})
else:
    print("Mapping file unavailable: using documented exclusion codes; obtain official mapping before final submission.")

before = len(df)
df = df.loc[df["discharge_disposition_id"].notna()].copy()
record("missing discharge disposition", before, len(df))
before = len(df)
df = df.loc[~df["discharge_disposition_id"].isin(excluded_codes)].copy()
record("death or hospice disposition", before, len(df))

before = len(df)
valid_care = df[CARE].notna().all(axis=1) & df["time_in_hospital"].between(1,14).fillna(False)
df = df.loc[valid_care].copy()
record("missing care dimensions or stay outside documented 1–14 days", before, len(df))
if df.empty:
    raise ValueError("No eligible encounters remain.")
df["readmitted_30"] = df["readmitted"].eq("<30").astype(int)
# No display of patient rows or identifiers in saved outputs.
display(pd.DataFrame(flow))
print("Eligible unique patients:", df["patient_nbr"].nunique())

Death/hospice mapping verified: {11: 'Expired', 13: 'Hospice / home', 14: 'Hospice / medical facility', 19: 'Expired at home. Medicaid only, hospice.', 20: 'Expired in a medical facility. Medicaid only, hospice.', 21: 'Expired, place unknown. Medicaid only, hospice.'}


,stage,removed,remaining
0,raw,0,101766
1,exact duplicate rows,0,101766
2,missing discharge disposition,0,101766
3,death or hospice disposition,2423,99343
4,missing care dimensions or stay outside docume...,0,99343


Eligible unique patients: 69990


## 3. Deterministic clinical feature preparation
Group ICD-9 diagnosis codes into broad chapters, preserving missing and invalid values separately. This is a predefined engineering decision, not a severity score and not a clinical diagnosis. Preserve the four care counts unchanged. Keep literal `None` (test not performed) distinct from missing in laboratory results. Treat integer admission codes as nominal categories. Do not infer gender or race from other columns. Keep missing demographics as missing until training preprocessing; fairness audits report them separately.

The clinical baseline uses recorded diagnoses (which may be coded during the encounter), demographics, admission information and prior-year utilisation. It is a **discharge-time** benchmark, not an admission-time prediction. `A1Cresult`, `max_glu_serum`, `change`, `diabetesMed` and disposition are retained for audit/sensitivity only, not used as clinical baseline predictors because they may reflect care decisions. Weight, payer, specialty, individual drug codes and unused columns are excluded from the primary experiment to limit scope; this is a proposed choice, not a data-driven result.

In [7]:
def icd9_chapter(value):
    if pd.isna(value):
        return "Missing"
    code = str(value).strip().upper()
    if re.fullmatch(r"E[0-9]{3}(?:\.[0-9]+)?", code):
        return "External causes (E)"
    if re.fullmatch(r"V[0-9]{2}(?:\.[0-9]+)?", code):
        return "Supplementary factors (V)"
    if not re.fullmatch(r"[0-9]{1,3}(?:\.[0-9]+)?", code):
        return "Unrecognised"
    n = int(code.split(".")[0])
    if n == 250:
        return "Diabetes"
    chapters = [(1,139,"Infectious"),(140,239,"Neoplasms"),(240,279,"Endocrine/metabolic"),
                (280,289,"Blood"),(290,319,"Mental health"),(320,389,"Nervous/sense organs"),
                (390,459,"Circulatory"),(460,519,"Respiratory"),(520,579,"Digestive"),
                (580,629,"Genitourinary"),(630,679,"Pregnancy/childbirth"),(680,709,"Skin"),
                (710,739,"Musculoskeletal"),(740,759,"Congenital"),(760,779,"Perinatal"),
                (780,799,"Symptoms/signs"),(800,999,"Injury/poisoning")]
    return next((label for lo,hi,label in chapters if lo <= n <= hi), "Unrecognised")

DIAG_GROUPS = [f"{c}_group" for c in DIAG]
for col, new in zip(DIAG, DIAG_GROUPS):
    df[new] = df[col].map(icd9_chapter)
for col in ADMISSION:
    df[col] = df[col].astype("string")

CLINICAL = DEMOGRAPHICS + ADMISSION + DIAG_GROUPS + COUNTS
AUDIT = ["discharge_disposition_id", "A1Cresult", "max_glu_serum", "change", "diabetesMed"]
TARGET = "readmitted_30"
KEEP = IDS + CLINICAL + CARE + AUDIT + [TARGET]
cohort = df[KEEP].sort_values("encounter_id").reset_index(drop=True)
# Fixed schema (no fitting or imputation on full data).
assert cohort["encounter_id"].is_unique
assert set(CLINICAL).isdisjoint(CARE + IDS + AUDIT + [TARGET])
print("Prepared shape:", cohort.shape)

Prepared shape: (99343, 24)


## 4. Freeze patient-grouped partitions
Use five stratified group folds, with fold 0 assigned to test, fold 1 to validation, and folds 2–4 to training (approximately 60/20/20 by encounter count). Stratification is approximate because patients can have several encounters. Fixed seed and fold assignments are recorded; do not change them in response to model scores. Do not use patient IDs as predictors. Check both classes exist in each split; if they do not, stop and review the design, not tune using test performance. Repeated patients are retained and later statistical inference must account for within-patient dependence.

In [8]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
if cohort["patient_nbr"].nunique() < 5:
    raise ValueError("Too few unique patients for five patient-grouped folds.")
fold = np.full(len(cohort), -1, dtype=int)
for i, (_, idx) in enumerate(splitter.split(cohort[CLINICAL], cohort[TARGET], groups=cohort["patient_nbr"])):
    fold[idx] = i
assert (fold >= 0).all()
cohort["split"] = np.select([fold == 0, fold == 1], ["test", "validation"], default="train")
assert cohort.groupby("patient_nbr")["split"].nunique().max() == 1
for name in ["train", "validation", "test"]:
    part = cohort.loc[cohort["split"].eq(name)]
    if set(part[TARGET]) != {0,1}:
        raise ValueError(f"Both target classes must exist in {name}.")
# Show partition sizes, not detailed held-out outcome distributions.
display(cohort.groupby("split").agg(encounters=("encounter_id","size"), patients=("patient_nbr","nunique")))
train = cohort.loc[cohort["split"].eq("train")]
print("Training early-readmission prevalence:", round(train[TARGET].mean(),4))

,encounters,patients
split,,
test,19868,14003
train,59604,42031
validation,19871,13956


Training early-readmission prevalence: 0.1139


## 5. Training-only audit and feature contract
Audit missingness in the training split. Do not learn preprocessing parameters from validation or test. For the later classifiers: median imputation for numeric features, explicit missing category and one-hot encoding for nominal features (unknown categories handled at transform time), and scale where needed. Fit these inside each training fold. Race/gender inclusion is a proposed primary experiment choice; compare a sensitivity version excluding them, and report subgroup errors regardless. No SMOTE in this notebook; consider class weighting or training-fold-only resampling later.

Feature sets: A = clinical; B = A + four observed care counts; C = B + four standardised signed residuals (and optional prespecified composite, separately evaluated). Expected-care estimators predict each observed care dimension using A only. Out-of-fold predictions on training patients are needed; if tuning classifiers by CV, repeat all feature generation within outer training folds. No discordance features are generated here.

In [9]:
audit = pd.DataFrame({"missing_count":train.isna().sum(), "missing_pct":100*train.isna().mean()})
display(audit.sort_values("missing_pct", ascending=False).round(2))
for col in LABS:
    print(f"Training {col}: literal None={train[col].eq('None').sum()}, missing={train[col].isna().sum()}")
feature_contract = {
    "target": TARGET, "group": "patient_nbr", "id": "encounter_id", "partition": "split",
    "clinical_features": CLINICAL, "clinical_numeric": COUNTS,
    "clinical_categorical": DEMOGRAPHICS + ADMISSION + DIAG_GROUPS,
    "observed_care_features": CARE, "audit_only": AUDIT,
    "feature_set_A": CLINICAL, "feature_set_B": CLINICAL + CARE,
    "planned_discordance_features": [f"discordance_{c}" for c in CARE],
    "feature_set_C_planned": CLINICAL + CARE + [f"discordance_{c}" for c in CARE],
    "prediction_time": "discharge", "seed": SEED,
    "excluded_dispositions": sorted(excluded_codes), "mapping_verified": bool(disposition),
    "cohort_rule": "Exclude missing disposition, death/hospice, missing observed care and stay outside 1–14 days. Retain repeat encounters.",
    "source_sha256": sha256(RAW), "mapping_sha256": sha256(map_path) if map_path else None,
    "raw_rows": len(raw), "eligible_rows": len(cohort),
    "versions": {"python":platform.python_version(), "pandas":pd.__version__, "numpy":np.__version__, "sklearn":sklearn.__version__},
}

,missing_count,missing_pct
race,1392,2.34
encounter_id,0,0.00
number_outpatient,0,0.00
readmitted_30,0,0.00
diabetesMed,0,0.00
change,0,0.00
max_glu_serum,0,0.00
A1Cresult,0,0.00
discharge_disposition_id,0,0.00
num_medications,0,0.00


Training A1Cresult: literal None=49427, missing=0
Training max_glu_serum: literal None=56486, missing=0


## 6. Export reproducible artefacts
CSV files are local, row-level research artefacts. Keep them out of public Git and public notebook outputs. Models should load the matching CSV with `keep_default_na=False, na_values=["?", ""]` and identifiers as strings. The separate validation notebook checks these files against the source and contract. Run validation before modelling. Re-running cleaning replaces outputs; do not regenerate after viewing held-out scores without documenting a new experiment.

In [10]:
paths = []
for name in ["train", "validation", "test"]:
    p = OUT / f"{name}.csv"
    cohort.loc[cohort["split"].eq(name)].to_csv(p, index=False)
    paths.append(p)
cohort[IDS + ["split"]].to_csv(OUT / "split_assignments.csv", index=False)
paths.append(OUT / "split_assignments.csv")
pd.DataFrame(flow).to_csv(OUT / "cohort_flow.csv", index=False)
paths.append(OUT / "cohort_flow.csv")
audit.to_csv(OUT / "training_missingness.csv", index_label="feature")
paths.append(OUT / "training_missingness.csv")
feature_contract["output_sha256"] = {p.name:sha256(p) for p in paths}
(OUT / "feature_contract.json").write_text(json.dumps(feature_contract, indent=2), encoding="utf-8")
print("Saved:", ", ".join(p.name for p in paths), "and feature_contract.json")
print("Next: run cleaning/validation.ipynb")

Saved: train.csv, validation.csv, test.csv, split_assignments.csv, cohort_flow.csv, training_missingness.csv and feature_contract.json
Next: run cleaning/validation.ipynb
